In [1]:
from pathlib import Path
import xarray as xr
import sys

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.gas_attenuation import specific_gas_attenuation, alpha_field

In [2]:
Processed_Dataset = "20261001_pallas_interpolation_CLARA"

In [3]:
in_path = Path(f"../data/processed/{Processed_Dataset}/aligned_LWP2.nc")
ds = xr.open_dataset(in_path)

In [4]:
alpha_W = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=94.0,
)

alpha_Ka = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=35.0,
)

In [5]:
print(alpha_W)
print(alpha_Ka)

print(f"W-band:  {float(alpha_W.min()):.3f} - {float(alpha_W.max()):.3f} dB/km")
print(f"Ka-band: {float(alpha_Ka.min()):.3f} - {float(alpha_Ka.max()):.3f} dB/km")

<xarray.DataArray 'alpha_94GHz' (Time: 30216, range: 442)> Size: 107MB
array([[0.38135125, 0.37840546, 0.37546296, ...,        nan,        nan,
               nan],
       [0.38129209, 0.37834675, 0.3754047 , ...,        nan,        nan,
               nan],
       [0.3812332 , 0.37828831, 0.3753467 , ...,        nan,        nan,
               nan],
       ...,
       [0.43237337, 0.43033825, 0.4282957 , ...,        nan,        nan,
               nan],
       [0.43237337, 0.43033825, 0.4282957 , ...,        nan,        nan,
               nan],
       [0.43237337, 0.43033825, 0.4282957 , ...,        nan,        nan,
               nan]], shape=(30216, 442))
Coordinates:
  * Time     (Time) datetime64[ns] 242kB 2026-10-01T00:43:14.052958488 ... 20...
  * range    (range) float64 4kB 104.3 119.3 134.2 ... 1.094e+04 1.097e+04
Attributes:
    units:      dB km-1
    long_name:  Gaseous attenuation at 94.0 GHz
    model:      ITU-R P.676
<xarray.DataArray 'alpha_35GHz' (Time: 30216, range

In [6]:
ds_alpha = xr.Dataset(
    {
        "alpha_W": alpha_W,
        "alpha_Ka": alpha_Ka,
    },
    attrs={
        "description": "Gas attenuation computed with ITU-R P.676-13",
        "created_by": "02_05_gas_attenuation.ipynb",
    },
)

ds_alpha.to_netcdf(f"../data/processed/{Processed_Dataset}/gas_attenuation.nc")